# Clustering Features for Household Energy Prediction

## Objective
Discover groups of similar observations and use cluster membership
as an additional feature for energy prediction.

## Datasets
- Model A: energy_model_A_pca11.csv
- Model B: energy_model_B_pca6_calendar.csv

Clustering will be performed separately for each dataset.

## Method
K-means groups observations by minimising squared distances from
their assigned cluster centres:

$$
\min_{\mu_1,\ldots,\mu_K}
\sum_{i=1}^{n}
\left\|\mathbf{x}_i-\mu_{c_i}\right\|^2
$$

where:
- K is the number of clusters.
- x_i is an observation's input feature vector.
- c_i is its assigned cluster.
- μ represents a cluster centre.

## Preventing Data Leakage
- Exclude Total_Energy, date, and split from clustering inputs.
- Fit scaling and K-means using training observations only.
- Assign test observations to the learned cluster centres.
- Preserve the existing chronological train/test split.

## Prediction Experiment
Encode cluster membership as categorical indicators and compare
prediction performance with and without these additional features.

Cluster numbers are labels, not ordered numerical quantities.

In [12]:
# File paths and data handling
from pathlib import Path
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt

# Clustering and preprocessing
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Cluster evaluation
from sklearn.metrics import silhouette_score
from sklearn.decomposition import KernelPCA

In [2]:
# Load both datasets
data_folder = Path("../data")

df_A = pd.read_csv(
    data_folder / "energy_model_A_pca11.csv",
    parse_dates=["date"]
)

df_B = pd.read_csv(
    data_folder / "energy_model_B_pca6_calendar.csv",
    parse_dates=["date"]
)

# Arrange observations chronologically
df_A = df_A.sort_values("date").reset_index(drop=True)
df_B = df_B.sort_values("date").reset_index(drop=True)

# Confirm matching observations and partitions
assert df_A[["date", "Total_Energy", "split"]].equals(
    df_B[["date", "Total_Energy", "split"]]
), "The datasets do not align."

# Summarise both datasets
display(pd.DataFrame({
    "Model": ["A", "B"],
    "Rows": [len(df_A), len(df_B)],
    "Training rows": [
        df_A["split"].eq("train").sum(),
        df_B["split"].eq("train").sum()
    ],
    "Testing rows": [
        df_A["split"].eq("test").sum(),
        df_B["split"].eq("test").sum()
    ],
    "Missing values": [
        df_A.isna().sum().sum(),
        df_B.isna().sum().sum()
    ]
}))

,Model,Rows,Training rows,Testing rows,Missing values
0,A,3289,2631,658,0
1,B,3289,2631,658,0


## Step 2: Prepare and Scale K-means Inputs

Exclude date, Total_Energy, and split from the clustering features.

Fit a separate StandardScaler on each model's training inputs, then
apply it to the corresponding test inputs.

Remove features that are constant during training because they cannot
help distinguish training observations.

In [3]:
exclude_columns = ["date", "Total_Energy", "split"]

clustering_data = {}
preparation_rows = []

for model_name, df in [("A", df_A), ("B", df_B)]:
    # Separate inputs using the existing chronological split
    X_train = df.loc[df["split"] == "train"].drop(columns=exclude_columns)
    X_test = df.loc[df["split"] == "test"].drop(columns=exclude_columns)

    # Identify useful clustering features from training data only
    constant_columns = X_train.columns[
        X_train.nunique() <= 1
    ].tolist()

    X_train = X_train.drop(columns=constant_columns)
    X_test = X_test.drop(columns=constant_columns)

    # Learn scaling from training inputs only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Store inputs and fitted scaler for later steps
    clustering_data[model_name] = {
        "features": X_train.columns.tolist(),
        "scaler": scaler,
        "X_train_scaled": X_train_scaled,
        "X_test_scaled": X_test_scaled,
        "train_index": X_train.index,
        "test_index": X_test.index
    }

    preparation_rows.append({
        "Model": model_name,
        "Training shape": X_train_scaled.shape,
        "Testing shape": X_test_scaled.shape,
        "Removed constant columns": ", ".join(constant_columns) or "None"
    })

display(pd.DataFrame(preparation_rows))

,Model,Training shape,Testing shape,Removed constant columns
0,A,"(2631, 11)","(658, 11)",None
1,B,"(2631, 14)","(658, 14)",month_5


## Step 3: Compare Candidate Cluster Counts

Fit K-means with K from 2 to 8 using training inputs only.

Record:
- Inertia: Total squared distance to assigned cluster centres.
- Silhouette score: Measures cluster separation.
- Smallest cluster size: Helps identify very small groups.

Use these results to investigate suitable cluster counts.

In [4]:
kmeans_candidates = {}
cluster_search_rows = []

for model_name, prepared in clustering_data.items():
    X_train_scaled = prepared["X_train_scaled"]

    for k in range(2, 9):
        kmeans = KMeans(
            n_clusters=k,
            init="k-means++",
            n_init=20,
            random_state=42
        )

        labels = kmeans.fit_predict(X_train_scaled)
        cluster_sizes = np.bincount(labels, minlength=k)

        kmeans_candidates[(model_name, k)] = kmeans

        cluster_search_rows.append({
            "Model": model_name,
            "K": k,
            "Inertia": kmeans.inertia_,
            "Silhouette": silhouette_score(X_train_scaled, labels),
            "Smallest cluster": cluster_sizes.min(),
            "Largest cluster": cluster_sizes.max()
        })

cluster_search_results = pd.DataFrame(cluster_search_rows)
display(cluster_search_results.round(3))

,Model,K,Inertia,Silhouette,Smallest cluster,Largest cluster
0,A,2,26800.959,0.102,851,1780
1,A,3,24936.645,0.101,506,1198
2,A,4,23376.654,0.100,486,862
3,A,5,21912.855,0.109,208,763
4,A,6,20843.244,0.113,194,694
5,A,7,19892.674,0.118,178,629
6,A,8,19052.381,0.119,174,518
7,B,2,32163.436,0.219,438,2193
8,B,3,28242.029,0.176,704,1183
9,B,4,24741.207,0.198,120,1127


In [5]:
df_fe = pd.read_csv(data_folder / "energy_1hour_fe_ready.csv")

print("Dataset shape:", df_fe.shape)
print("\nColumns:")
print(df_fe.columns.tolist())

display(df_fe.head())

print("\nMissing values:")
display(df_fe.isna().sum().loc[lambda counts: counts > 0])

if "split" in df_fe.columns:
    print("\nExisting split counts:")
    display(df_fe["split"].value_counts())

Dataset shape: (3289, 43)

Columns:
['date', 'T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'RH_1', 'RH_2', 'RH_3', 'RH_4', 'RH_5', 'RH_6', 'RH_7', 'RH_8', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'day_of_week', 'is_weekend', 'Floor_1_Temp', 'Floor_2_Temp', 'Floor_1_Humidity', 'Floor_2_Humidity', 'hour_sin', 'hour_cos', 'is_afternoon_peak', 'month_2', 'month_3', 'month_4', 'month_5', 'Floor_1_Temp_Change', 'Floor_2_Temp_Change', 'Both_Floors_Warming', 'Weekend_Afternoon', 'Total_Energy']


,date,T1,T2,T3,T4,T5,T6,T7,T8,T9,...,is_afternoon_peak,month_2,month_3,month_4,month_5,Floor_1_Temp_Change,Floor_2_Temp_Change,Both_Floors_Warming,Weekend_Afternoon,Total_Energy
0,2016-01-11 17:00:00,19.890000,19.200000,19.790000,18.932778,17.166667,6.586667,17.177778,18.150000,17.016667,...,1,0,0,0,0,0.000000,0.000000,0,0,540
1,2016-01-11 18:00:00,19.897778,19.268889,19.770000,18.908333,17.111111,6.180000,17.229583,18.094444,16.981667,...,1,0,0,0,0,0.018889,-0.023750,0,0,1370
2,2016-01-11 19:00:00,20.495556,19.925556,20.052222,18.969444,17.136111,5.857361,17.850000,18.156111,16.902222,...,0,0,0,0,0,0.512222,0.137750,1,0,1190
3,2016-01-11 20:00:00,20.961111,20.251111,20.213889,19.190833,17.615556,5.469444,17.632222,18.773333,16.890000,...,0,0,0,0,0,0.317593,0.217611,1,0,960
4,2016-01-11 21:00:00,21.311667,20.587778,20.373333,19.425556,18.427222,5.578889,17.863611,19.153333,16.890000,...,0,0,0,0,0,0.282222,0.331556,1,0,760



Missing values:


Series([], dtype: int64)

In [6]:
# Prepare timestamps
df_fe["date"] = pd.to_datetime(df_fe["date"])
df_fe = df_fe.sort_values("date").reset_index(drop=True)

# Recover the existing split by matching timestamps
df_kernel = df_fe.merge(
    df_A[["date", "split"]],
    on="date",
    how="left",
    validate="one_to_one"
)

assert df_kernel["split"].notna().all(), "Some timestamps have no split."
assert len(df_kernel) == len(df_A), "Dataset row counts differ."

# Separate inputs
kernel_features = df_kernel.drop(
    columns=["date", "Total_Energy", "split"]
)

kernel_train_mask = df_kernel["split"].eq("train")
kernel_test_mask = df_kernel["split"].eq("test")

X_kernel_train = kernel_features.loc[kernel_train_mask].copy()
X_kernel_test = kernel_features.loc[kernel_test_mask].copy()

# Remove constant columns based on training data only
constant_features = X_kernel_train.columns[
    X_kernel_train.nunique() <= 1
].tolist()

X_kernel_train = X_kernel_train.drop(columns=constant_features)
X_kernel_test = X_kernel_test.drop(columns=constant_features)

# Fit scaling on training observations only
kernel_scaler = StandardScaler()

X_kernel_train_scaled = kernel_scaler.fit_transform(X_kernel_train)
X_kernel_test_scaled = kernel_scaler.transform(X_kernel_test)

print("Training shape:", X_kernel_train_scaled.shape)
print("Testing shape:", X_kernel_test_scaled.shape)
print("Removed constant features:", constant_features)

Training shape: (2631, 40)
Testing shape: (658, 40)
Removed constant features: ['month_5']


## Step 6: Apply RBF Kernel PCA

Transform the 40 scaled input features into 10 nonlinear components.

Initial settings:
- Kernel: RBF
- Gamma: 0.025
- Components: 10

Fit Kernel PCA on training observations only, then transform the test
observations using the fitted model.

These initial settings will be evaluated before selecting a final
representation for K-means.

In [9]:
# Initial Kernel PCA settings
kernel_gamma = 1 / X_kernel_train_scaled.shape[1]

kpca = KernelPCA(
    n_components=10,
    kernel="rbf",
    gamma=kernel_gamma,
    eigen_solver="arpack",
    random_state=42
)

# Fit using training observations only
X_kpca_train = kpca.fit_transform(X_kernel_train_scaled)
X_kpca_test = kpca.transform(X_kernel_test_scaled)

print("Gamma:", kernel_gamma)
print("Kernel PCA training shape:", X_kpca_train.shape)
print("Kernel PCA testing shape:", X_kpca_test.shape)

display(pd.DataFrame(
    X_kpca_train[:5],
    columns=[f"KPC{i}" for i in range(1, 11)]
))

Gamma: 0.025
Kernel PCA training shape: (2631, 10)
Kernel PCA testing shape: (658, 10)


,KPC1,KPC2,KPC3,KPC4,KPC5,KPC6,KPC7,KPC8,KPC9,KPC10
0,0.416361,-0.117750,-0.106100,0.060854,0.169728,0.088797,-0.063756,-0.078998,-0.106595,-0.090707
1,0.432087,-0.112151,-0.096887,0.068222,0.171257,0.087797,-0.064268,-0.091052,-0.142589,-0.106814
2,0.339233,-0.164646,-0.158153,0.036196,0.075276,0.095098,0.004826,-0.063868,0.047411,-0.140957
3,0.353531,-0.182585,-0.137351,-0.001617,0.023407,0.099684,-0.002482,-0.046016,0.059528,-0.135701
4,0.348559,-0.192574,-0.114181,-0.056209,0.012097,0.056021,-0.006431,-0.039121,0.054823,-0.155175


## Step 7: Evaluate K-means After Kernel PCA

Fit K-means on the training Kernel PCA components for K = 2 to 8.

Compare silhouette scores and cluster sizes. Higher silhouette scores
indicate better separation within this representation, but do not
guarantee better energy predictions.

Scores from different feature representations describe different
distance spaces, so comparisons should be interpreted cautiously.

In [11]:
kpca_kmeans_candidates = {}
kpca_cluster_rows = []

for k in range(2, 9):
    kmeans = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        random_state=42
    )

    labels = kmeans.fit_predict(X_kpca_train)
    cluster_sizes = np.bincount(labels, minlength=k)

    kpca_kmeans_candidates[k] = kmeans

    kpca_cluster_rows.append({
        "K": k,
        "Inertia": kmeans.inertia_,
        "Silhouette": silhouette_score(X_kpca_train, labels),
        "Smallest cluster": cluster_sizes.min(),
        "Largest cluster": cluster_sizes.max()
    })

kpca_cluster_results = pd.DataFrame(kpca_cluster_rows)
display(kpca_cluster_results.round(3))

,K,Inertia,Silhouette,Smallest cluster,Largest cluster
0,2,704.164,0.177,1230,1401
1,3,584.173,0.210,725,1020
2,4,517.649,0.215,536,778
3,5,466.900,0.220,267,769
4,6,423.965,0.220,268,531
5,7,391.727,0.228,263,480
6,8,368.007,0.233,192,462


## Step 8: Prepare Model B Environmental Features for Kernel PCA

Use the same six environmental inputs used for Model B's ordinary PCA:

- Floor_1_Temp
- Floor_2_Temp
- Floor_1_Humidity
- Floor_2_Humidity
- T_out
- RH_out

Fit scaling using training observations only.
Calendar features are excluded from this transformation.

In [13]:
# Load original features and existing split labels
data_folder = Path("../data")

df_fe = pd.read_csv(
    data_folder / "energy_1hour_fe_ready.csv",
    parse_dates=["date"]
)

split_reference = pd.read_csv(
    data_folder / "energy_model_B_pca6_calendar.csv",
    usecols=["date", "split"],
    parse_dates=["date"]
)

# Match observations by timestamp
df_kernel = df_fe.merge(
    split_reference,
    on="date",
    how="left",
    validate="one_to_one"
).sort_values("date").reset_index(drop=True)

assert df_kernel["split"].notna().all(), "Some timestamps have no split."
assert len(df_kernel) == len(split_reference), "Dataset row counts differ."

# Select the six environmental features
kernel_B_features = [
    "Floor_1_Temp",
    "Floor_2_Temp",
    "Floor_1_Humidity",
    "Floor_2_Humidity",
    "T_out",
    "RH_out"
]

kernel_train_mask = df_kernel["split"].eq("train")
kernel_test_mask = df_kernel["split"].eq("test")

X_kernel_B_train = df_kernel.loc[
    kernel_train_mask, kernel_B_features
].copy()

X_kernel_B_test = df_kernel.loc[
    kernel_test_mask, kernel_B_features
].copy()

assert not X_kernel_B_train.isna().any().any(), "Training inputs contain missing values."
assert not X_kernel_B_test.isna().any().any(), "Test inputs contain missing values."

# Learn scaling from training observations only
kernel_B_scaler = StandardScaler()

X_kernel_B_train_scaled = kernel_B_scaler.fit_transform(
    X_kernel_B_train
)
X_kernel_B_test_scaled = kernel_B_scaler.transform(
    X_kernel_B_test
)

print("Training shape:", X_kernel_B_train_scaled.shape)
print("Testing shape:", X_kernel_B_test_scaled.shape)

display(X_kernel_B_train.head())

Training shape: (2631, 6)
Testing shape: (658, 6)


,Floor_1_Temp,Floor_2_Temp,Floor_1_Humidity,Floor_2_Humidity,T_out,RH_out
0,19.626667,17.688778,45.342361,47.282528,6.308333,92.000000
1,19.645556,17.665028,45.060417,47.395694,5.941667,91.583333
2,20.157778,17.802778,48.698194,48.645583,6.000000,89.750000
3,20.475370,18.020389,47.118287,52.298028,6.000000,87.583333
4,20.757593,18.351944,45.631296,50.985000,5.833333,87.416667


In [15]:
# Extract targets using the same masks as the input features
y_kernel_B_train = df_kernel.loc[
    kernel_train_mask, "Total_Energy"
].copy()

y_kernel_B_test = df_kernel.loc[
    kernel_test_mask, "Total_Energy"
].copy()

# Verify alignment and completeness
assert X_kernel_B_train.index.equals(y_kernel_B_train.index)
assert X_kernel_B_test.index.equals(y_kernel_B_test.index)
assert y_kernel_B_train.notna().all()
assert y_kernel_B_test.notna().all()

print("X training shape:", X_kernel_B_train.shape)
print("y training shape:", y_kernel_B_train.shape)
print("X testing shape:", X_kernel_B_test.shape)
print("y testing shape:", y_kernel_B_test.shape)

display(
    X_kernel_B_train.join(y_kernel_B_train).head()
)

X training shape: (2631, 6)
y training shape: (2631,)
X testing shape: (658, 6)
y testing shape: (658,)


,Floor_1_Temp,Floor_2_Temp,Floor_1_Humidity,Floor_2_Humidity,T_out,RH_out,Total_Energy
0,19.626667,17.688778,45.342361,47.282528,6.308333,92.000000,540
1,19.645556,17.665028,45.060417,47.395694,5.941667,91.583333,1370
2,20.157778,17.802778,48.698194,48.645583,6.000000,89.750000,1190
3,20.475370,18.020389,47.118287,52.298028,6.000000,87.583333,960
4,20.757593,18.351944,45.631296,50.985000,5.833333,87.416667,760


## Step 10: Apply Kernel PCA to Model B Environmental Inputs

Transform the six scaled environmental features using RBF Kernel PCA.

Initial settings:
- Components: 6
- Gamma: 1/6

Fit using training inputs only, then transform test inputs.
Total_Energy is excluded from the transformation.

In [16]:
# Initial RBF Kernel PCA for the six environmental inputs
kernel_B_gamma = 1 / X_kernel_B_train_scaled.shape[1]

kpca_B = KernelPCA(
    n_components=6,
    kernel="rbf",
    gamma=kernel_B_gamma,
    eigen_solver="arpack",
    random_state=42
)

X_kpca_B_train = kpca_B.fit_transform(X_kernel_B_train_scaled)
X_kpca_B_test = kpca_B.transform(X_kernel_B_test_scaled)

# Keep timestamps' row alignment in the transformed DataFrames
kpca_B_columns = [f"KPC{i}" for i in range(1, 7)]

df_kpca_B_train = pd.DataFrame(
    X_kpca_B_train,
    columns=kpca_B_columns,
    index=X_kernel_B_train.index
)

df_kpca_B_test = pd.DataFrame(
    X_kpca_B_test,
    columns=kpca_B_columns,
    index=X_kernel_B_test.index
)

print("Gamma:", kernel_B_gamma)
print("Training shape:", df_kpca_B_train.shape)
print("Testing shape:", df_kpca_B_test.shape)

# Display target alongside components for inspection only
display(df_kpca_B_train.join(y_kernel_B_train).head())

Gamma: 0.16666666666666666
Training shape: (2631, 6)
Testing shape: (658, 6)


,KPC1,KPC2,KPC3,KPC4,KPC5,KPC6,Total_Energy
0,0.338036,-0.508936,-0.306148,-0.200607,-0.315349,0.192132,540
1,0.326923,-0.520384,-0.304148,-0.192763,-0.331524,0.194298,1370
2,0.366854,-0.361503,-0.298205,-0.179377,-0.041243,0.178447,1190
3,0.421603,-0.389655,-0.286654,-0.221156,0.078362,0.183226,960
4,0.477991,-0.469569,-0.198093,-0.270283,0.083940,0.159896,760


## Step 11: Evaluate K-means on Model B Kernel Components

Compare cluster counts from K = 2 to 8 using training observations.

Evaluate:
- Silhouette score: Higher values indicate clearer separation.
- Cluster sizes: Check whether any groups contain very few observations.
- Inertia: Within-cluster squared distances; normally decreases as K increases.

These measures assess clustering structure, not energy prediction accuracy.

In [17]:
kpca_B_kmeans_candidates = {}
kpca_B_cluster_rows = []

for k in range(2, 9):
    kmeans = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        random_state=42
    )

    # Use kernel components only, excluding Total_Energy
    labels = kmeans.fit_predict(X_kpca_B_train)
    cluster_sizes = np.bincount(labels, minlength=k)

    kpca_B_kmeans_candidates[k] = kmeans

    kpca_B_cluster_rows.append({
        "K": k,
        "Inertia": kmeans.inertia_,
        "Silhouette": silhouette_score(X_kpca_B_train, labels),
        "Smallest cluster": cluster_sizes.min(),
        "Largest cluster": cluster_sizes.max()
    })

kpca_B_cluster_results = pd.DataFrame(kpca_B_cluster_rows)
display(kpca_B_cluster_results.round(3))

,K,Inertia,Silhouette,Smallest cluster,Largest cluster
0,2,885.617,0.217,1244,1387
1,3,683.422,0.268,753,1016
2,4,559.690,0.284,618,705
3,5,460.159,0.312,457,591
4,6,394.508,0.322,320,567
5,7,347.826,0.319,257,518
6,8,315.567,0.315,274,422


In [18]:
# Reuse the already fitted K = 6 candidate
selected_k_B = 6
selected_kmeans_B = kpca_B_kmeans_candidates[selected_k_B]

# Build a training-only interpretation table
cluster_profile_B = X_kernel_B_train.copy()
cluster_profile_B["Cluster"] = selected_kmeans_B.labels_
cluster_profile_B["Total_Energy"] = y_kernel_B_train

cluster_summary_B = cluster_profile_B.groupby("Cluster").agg(
    Hours=("Total_Energy", "size"),
    Mean_Energy_Wh=("Total_Energy", "mean"),
    Median_Energy_Wh=("Total_Energy", "median"),
    Floor_1_Temp=("Floor_1_Temp", "mean"),
    Floor_2_Temp=("Floor_2_Temp", "mean"),
    Floor_1_Humidity=("Floor_1_Humidity", "mean"),
    Floor_2_Humidity=("Floor_2_Humidity", "mean"),
    T_out=("T_out", "mean"),
    RH_out=("RH_out", "mean")
)

display(cluster_summary_B.round(2))

,Hours,Mean_Energy_Wh,Median_Energy_Wh,Floor_1_Temp,Floor_2_Temp,Floor_1_Humidity,Floor_2_Humidity,T_out,RH_out
Cluster,,,,,,,,,
0,343,570.52,290.0,19.59,17.84,43.52,47.08,5.97,88.57
1,567,516.91,370.0,21.12,20.45,39.96,41.56,4.58,90.25
2,433,759.91,530.0,22.31,21.39,39.76,40.50,11.20,67.15
3,456,715.86,530.0,20.69,19.97,36.64,37.10,5.94,69.20
4,320,662.75,450.0,21.66,19.80,44.15,48.57,8.75,87.09
5,512,505.90,320.0,19.48,18.79,38.17,40.06,0.79,88.39


In [19]:
# Calendar features used in Model B
calendar_features = [
    "hour_sin",
    "hour_cos",
    "is_weekend",
    "is_afternoon_peak",
    "month_2",
    "month_3",
    "month_4",
    "month_5",
    "Weekend_Afternoon"
]

# Recover training assignments and predict test assignments
train_clusters = pd.Series(
    selected_kmeans_B.labels_,
    index=X_kernel_B_train.index
)

test_clusters = pd.Series(
    selected_kmeans_B.predict(X_kpca_B_test),
    index=X_kernel_B_test.index
)

all_clusters = pd.concat(
    [train_clusters, test_clusters]
).sort_index()

# Encode cluster labels as categories; cluster 0 is the reference
cluster_indicators = pd.get_dummies(
    pd.Series(
        pd.Categorical(
            all_clusters,
            categories=range(selected_k_B)
        ),
        index=all_clusters.index,
        name="Cluster"
    ),
    prefix="Cluster",
    drop_first=True,
    dtype=int
)

# Combine training and testing kernel components in original row order
all_kernel_components = pd.concat(
    [df_kpca_B_train, df_kpca_B_test]
).sort_index()

assert all_kernel_components.index.equals(df_kernel.index)
assert cluster_indicators.index.equals(df_kernel.index)

# Assemble the prediction dataset
df_cluster_ready = pd.concat(
    [
        df_kernel[["date"]],
        all_kernel_components,
        df_kernel[calendar_features],
        cluster_indicators,
        df_kernel[["Total_Energy", "split"]]
    ],
    axis=1
)

assert not df_cluster_ready.isna().any().any(), (
    "The output contains missing values."
)

# Save CSV
cluster_csv_path = (
    data_folder / "energy_model_B_kpca6_calendar_clusters.csv"
)

df_cluster_ready.to_csv(cluster_csv_path, index=False)

print("Saved:", cluster_csv_path)
print("Shape:", df_cluster_ready.shape)
print("Columns:", df_cluster_ready.columns.tolist())
display(df_cluster_ready.head())

Saved: ..\data\energy_model_B_kpca6_calendar_clusters.csv
Shape: (3289, 23)
Columns: ['date', 'KPC1', 'KPC2', 'KPC3', 'KPC4', 'KPC5', 'KPC6', 'hour_sin', 'hour_cos', 'is_weekend', 'is_afternoon_peak', 'month_2', 'month_3', 'month_4', 'month_5', 'Weekend_Afternoon', 'Cluster_1', 'Cluster_2', 'Cluster_3', 'Cluster_4', 'Cluster_5', 'Total_Energy', 'split']


,date,KPC1,KPC2,KPC3,KPC4,KPC5,KPC6,hour_sin,hour_cos,is_weekend,...,month_4,month_5,Weekend_Afternoon,Cluster_1,Cluster_2,Cluster_3,Cluster_4,Cluster_5,Total_Energy,split
0,2016-01-11 17:00:00,0.338036,-0.508936,-0.306148,-0.200607,-0.315349,0.192132,-0.965926,-2.588190e-01,False,...,0,0,0,0,0,0,0,0,540,train
1,2016-01-11 18:00:00,0.326923,-0.520384,-0.304148,-0.192763,-0.331524,0.194298,-1.000000,-1.836970e-16,False,...,0,0,0,0,0,0,0,0,1370,train
2,2016-01-11 19:00:00,0.366854,-0.361503,-0.298205,-0.179377,-0.041243,0.178447,-0.965926,2.588190e-01,False,...,0,0,0,0,0,0,0,0,1190,train
3,2016-01-11 20:00:00,0.421603,-0.389655,-0.286654,-0.221156,0.078362,0.183226,-0.866025,5.000000e-01,False,...,0,0,0,0,0,0,0,0,960,train
4,2016-01-11 21:00:00,0.477991,-0.469569,-0.198093,-0.270283,0.083940,0.159896,-0.707107,7.071068e-01,False,...,0,0,0,0,0,0,0,0,760,train
